In [ ]:
import os
import sys
import jax
jax.config.update("jax_enable_x64", True) 

import jax.numpy as jnp
import numpy as np
import qutip as qt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.config import ExperimentConfig
from src.physics.operators import build_two_qubit_base, assemble_H0
from src.verification.qutip_device import HiddenDevice
from src.verification.plots import plot_absolute_cityscape
from src.control.sequences import  build_local_drive_operators, generate_bell_state, calibrate_virtual_z

# Load Configurations
config = ExperimentConfig.load("configs/two_qubit_CZ.yaml")
dt = config.pulse.dt_ns

# Build static JAX bases
H_anharm, H_int_base, Hz_control = build_two_qubit_base(levels=config.hardware.levels)
# Build the 9D single-qubit drive operators
Hx1, Hy1, Hx2, Hy2 = build_local_drive_operators(levels=config.hardware.levels)

# Load the optimized CZ pulse from the previous calibration step
cal_pulse_rad = np.load("../configs/cal_pulse_rad.npy")
print("Loaded optimized CZ pulse from ../configs/cal_pulse_rad.npy")

cal_g, cal_d, cal_a = np.load("../configs/cal_params.npy")
H0_cal = assemble_H0(H_anharm, H_int_base, Hz_control, cal_g, cal_d, cal_a)

In [ ]:

import matplotlib.pyplot as plt

duration_ns = 20.0

# Calibrate Virtual-Z phases on the Digital Twin (propagators computed once)
vz_cal = calibrate_virtual_z(
    duration_ns, dt, cal_a, cal_d, cal_pulse_rad, 
    H0_cal, Hx1, Hy1, Hx2, Hy2, Hz_control
)
c1, c2 = vz_cal["v_phase1"], vz_cal["v_phase2"]


print(f"Digital Twin Calibration -> Best Infidelity: {vz_cal['infidelity']*100:.2f}% (Ceiling Fidelity: {vz_cal['fidelity']*100:.2f}%)")
print(f"Calibrated Phases -> Q1 (tomography frame): {c1:5.2f} rad | Q2 (U3 drive frame): {c2:5.2f} rad\n")

# Plot the 2D Virtual-Z Calibration Landscape
plt.figure(figsize=(6, 4.5))
plt.contourf(
    vz_cal["grid_rad"], vz_cal["grid_rad"], vz_cal["fidelity_grid"].T, 
    levels=25, cmap="viridis"
)
plt.colorbar(label=r"$\Phi^+$ State Overlap")
plt.plot(c1, c2, "x", markersize=6, label=f"Optimum ({c1:.2f}, {c2:.2f} rad)")
plt.xlabel("Virtual-Z Phase Q1 (rad)")
plt.ylabel("Virtual-Z Phase Q2 (rad)")
plt.title("Digital Twin Virtual-Z Calibration Landscape")
plt.legend()
plt.tight_layout()
plt.show()

# Execute single-shot sequence on the Digital Twin (noiseless ceiling)
rho_corr = generate_bell_state(
    duration_ns, dt, cal_a, cal_d, cal_pulse_rad, 
    H0_cal, Hx1, Hy1, Hx2, Hy2, Hz_control,
    v_phase1=c1, v_phase2=c2
)

populations = np.abs(np.real(np.diag(rho_corr)))
print(f"Digital Twin Density Matrix (|rho|):\n{np.abs(rho_corr)}")
print(f"Populations: |00>: {populations[0]:.3f} | |01>: {populations[1]:.3f} | |10>: {populations[2]:.3f} | |11>: {populations[3]:.3f}")

fig = plot_absolute_cityscape(rho_corr)
fig.show()

In [ ]:
# Build the target Bell state in the 9D two-qutrit space
psi_full = (qt.tensor(qt.basis(3, 0), qt.basis(3, 0)) + qt.tensor(qt.basis(3, 1), qt.basis(3, 1))).unit()

# Extract the computational subspace indices [00, 01, 10, 11] -> indices [0, 1, 3, 4]
comp_indices = [0, 1, 3, 4]
psi_target_vec = psi_full.full().flatten()[comp_indices]
psi_target_vec = psi_target_vec / np.linalg.norm(psi_target_vec)

psi_target_qobj = qt.Qobj(psi_target_vec)
rho_target = psi_target_qobj * psi_target_qobj.dag()

rho_phys_qobj = qt.Qobj(rho_corr)
rho_mat = rho_phys_qobj.full()

# Mask phases where magnitude is near zero (< 1e-6) to avoid meaningless angle noise
masked_phases = np.where(np.abs(rho_mat) >= 1e-6, np.angle(rho_mat), 0.0)

print(f"Phases (rad, masked for |rho| < 1e-6):\n{np.round(masked_phases, 3)}\n")
print(f"Absolute values:\n{np.round(np.abs(rho_mat), 3)}\n")

fidelity_twin = qt.fidelity(rho_phys_qobj, rho_target) ** 2
print(f"Digital Twin Bell State Fidelity (Noiseless Ceiling): {fidelity_twin * 100:.2f}%")

In [ ]:
from src.control.sequences import apply_frame, pack_sequence, build_bell_waveforms


idx = [0, 1, 3, 4]
rho00 = qt.ket2dm(qt.tensor(qt.basis(3, 0), qt.basis(3, 0)))
psi_t = np.zeros(4, complex); psi_t[[0, 3]] = 1/np.sqrt(2)

def bell_fid(rho9, v1, v2):
    comp = apply_frame(rho9, v1, v2).full()[np.ix_(idx, idx)]   
    return float(np.real(psi_t.conj() @ comp @ psi_t))


fridge = HiddenDevice(true_coupling=config.hardware.coupling_mhz,
                      true_detuning=config.hardware.idle_detuning_mhz,
                      true_anharm=config.hardware.anharmonicity_mhz,
                      t1_ns=config.decoherence.t1_ns, t2_ns=config.decoherence.t2_ns)
assert abs(fridge.dt - dt) < 1e-12

wf = build_bell_waveforms(duration_ns, dt, cal_a, cal_d, cal_pulse_rad, v_phase2=c2)

rho_f = fridge.play_sequence(pack_sequence(wf), rho00)
F_B = bell_fid(rho_f, c1, c2)

# Population breakdown, to see where the loss is
p = np.real(np.diag(rho_f.full()))
print(f"B: fridge Bell fidelity (frozen twin phases): {F_B*100:.2f}%")
print(f"   P(comp. subspace) = {p[idx].sum():.4f}  (1 - this = leakage)")

In [ ]:
from src.verification.benchmarks import run_chsh_experiment

# Sanity Check on Ideal |Phi+> State
phi_ideal = (qt.tensor(qt.basis(3, 0), qt.basis(3, 0)) + qt.tensor(qt.basis(3, 1), qt.basis(3, 1))).unit()
S_ideal, err_ideal, _ = run_chsh_experiment(qt.ket2dm(phi_ideal), n_shots=5000, seed=42)
print(f"1. Ideal |Phi+> Sanity Check -> S = {S_ideal:.3f} ± {err_ideal:.3f} (Quantum max: {2*np.sqrt(2):.3f})")

# Fridge State WITH Virtual-Z Frame Correction
rho_framed = apply_frame(rho_f, c1, c2)
S_cal, err_cal, E_cal = run_chsh_experiment(rho_framed, n_shots=5000, seed=42)
sigma_violation = (S_cal - 2.0) / err_cal
print(f"2. Fridge (Calibrated Virtual-Z) -> S = {S_cal:.3f} ± {err_cal:.3f} ({sigma_violation:.1f}σ violation over classical bound 2.0)")

# Fridge State WITHOUT Virtual-Z Frame Correction (showing why phase tracking matters)
S_uncal, err_uncal, _ = run_chsh_experiment(rho_f, n_shots=5000, seed=42)
print(f"3. Fridge (Uncorrected Frame)    -> S = {S_uncal:.3f} ± {err_uncal:.3f}\n")

for k, v in E_cal.items():
    ideal_str = "-0.7071" if k == "E(a', b')" else "+0.7071"
    print(f"   {k:10s} = {v:+.4f}  (Ideal: {ideal_str})")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Left subplot: Individual Correlators E(a, b)
labels = list(E_cal.keys())
vals = list(E_cal.values())
ideal_vals = [1/np.sqrt(2), 1/np.sqrt(2), 1/np.sqrt(2), -1/np.sqrt(2)]

x = np.arange(len(labels))
ax1.bar(x - 0.18, ideal_vals, width=0.35, alpha=0.5, color="gray", label=r"Ideal $\Phi^+$ ($\pm 1/\sqrt{2}$)")
ax1.bar(x + 0.18, vals, width=0.35, color="teal", label="QuTiP Fridge (5,000 shots)")
ax1.axhline(0, color="black", linewidth=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.set_ylim(-1.0, 1.0)
ax1.set_ylabel("Parity Correlator")
ax1.set_title("CHSH Parity Correlators on Hardware")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right subplot: Total CHSH S-Value Comparison
categories = ["Uncorrected\nFrame", "Fridge +\nVirtual-Z", r"Ideal $\Phi^+$"]
s_values = [S_uncal, S_cal, S_ideal]
s_errors = [err_uncal, err_cal, err_ideal]
colors = ["purple", "teal", "royalblue"]

ax2.bar(categories, s_values, yerr=s_errors, capsize=6, color=colors, alpha=0.85, width=0.4)
ax2.axhline(2.0, color="darkorange", linestyle="--", linewidth=2, label="Classical Bound (S = 2.0)")
ax2.axhline(2 * np.sqrt(2), color="navy", linestyle=":", linewidth=2, label=r"Quantum Bound ($2\sqrt{2} \approx 2.828$)")
ax2.set_ylim(0, 3.2)
ax2.set_ylabel("CHSH Parameter S")
ax2.set_title("Bell Inequality Violation")
ax2.legend(loc="lower right")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()